# Wikidata Duplicate Detector

Created by [Matt Artz](https://www.mattartz.me/) | [GitHub](https://github.com/MattArtzAnthro) | [ORCID](https://orcid.org/0000-0002-3822-1429)

---

## What This Notebook Does

Detects duplicate scholarly article records in Wikidata using two methods:

- **DOI-based detection**: Finds items sharing the same DOI — definitive duplicates requiring merger
- **Title-based detection**: Finds items with matching normalized titles — candidates requiring manual review

Scopes queries to specific journals (21 AAA journals by default, configurable). Includes inspection tools to examine individual duplicate pairs.

**Endpoint note**: Uses the scholarly endpoint (`query-scholarly.wikidata.org`) by default, with optional fallback to the general endpoint.

**Important**: This notebook detects duplicates but does not automatically merge or delete records. All changes to Wikidata require manual review.

## Workflow

1. **Configure**: Select endpoint and journal set
2. **DOI detection**: Find items sharing DOIs across configured journals
3. **Title detection**: Find items with matching normalized titles per journal
4. **Inspect**: Examine specific duplicate pairs to understand divergence
5. **Export**: Download results as CSV for cleanup workflows

## Citation

If you use this notebook, please cite:

> Artz, M. (2026). Wikidata Duplicate Detector. GitHub. https://github.com/MattArtzAnthro/wikidata-tools

*A citable DOI will be available via Zenodo.*

## License

[CC BY-NC 4.0](https://creativecommons.org/licenses/by-nc/4.0/)

## Citation

If you use this notebook, please cite:

> Artz, Matt. (2026). MattArtzAnthro/wikidata-tools. Zenodo. https://doi.org/10.5281/zenodo.18912858

## License

[CC BY-NC 4.0](https://creativecommons.org/licenses/by-nc/4.0/)

## Setup and Package Installation

*Install required Python packages and import necessary libraries for SPARQL queries, data processing, and interactive widgets. Run this cell first to ensure all dependencies are available.*

In [ ]:
# Install required packages
!pip install SPARQLWrapper pandas ipywidgets

import pandas as pd
from SPARQLWrapper import SPARQLWrapper, JSON
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from datetime import datetime
import json
import os

# Wikidata SPARQL endpoints
ENDPOINTS = {
    "scholarly": "https://query-scholarly.wikidata.org/sparql",
    "general": "https://query.wikidata.org/sparql"
}

# Create output directory
OUTPUT_PATH = "/content/duplicate_detection_outputs/"
os.makedirs(OUTPUT_PATH, exist_ok=True)

print("✓ All packages installed successfully")
print(f"✓ Output directory created: {OUTPUT_PATH}")
print()
print("Available SPARQL endpoints:")
for name, url in ENDPOINTS.items():
    print(f"   • {name}: {url}")

## Endpoint and Journal Configuration

*Select the SPARQL endpoint and define journal QIDs to analyze. The scholarly endpoint is optimized for bibliographic queries and recommended for this use case. The default journal set includes AAA publications and related anthropology journals.*

In [ ]:
# Default AAA and anthropology journal QIDs with labels
DEFAULT_JOURNALS = {
    "Q4579783": "American Anthropologist",
    "Q15762644": "American Ethnologist",
    "Q27724868": "Annals of Anthropological Practice",
    "Q15754091": "Anthropology & Education Quarterly",
    "Q15755065": "Anthropology and Humanism",
    "Q4773909": "Anthropology of Consciousness",
    "Q15766859": "Anthropology of Work Review",
    "Q15759624": "Archeological Papers of the American Anthropological Association",
    "Q15752788": "City & Society",
    "Q63871858": "Culture, Agriculture, Food and Environment",
    "Q73541368": "Ecological and Environmental Anthropology",
    "Q15757300": "Ethos",
    "Q96701409": "General Anthropology",
    "Q5531693": "General Anthropology Bulletin",
    "Q73541374": "Journal for the Anthropology of North America",
    "Q15752869": "Journal of Latin American and Caribbean Anthropology",
    "Q15760155": "Journal of Linguistic Anthropology",
    "Q6806259": "Medical Anthropology Quarterly",
    "Q15752491": "Museum Anthropology",
    "Q63871800": "North American Dialogue",
    "Q27722507": "PoLAR: Political and Legal Anthropology Review"
}

class Config:
    """Configuration for duplicate detection"""
    JOURNALS = DEFAULT_JOURNALS.copy()
    OUTPUT_PATH = OUTPUT_PATH
    ACTIVE_ENDPOINT = "scholarly"  # Default to scholarly endpoint
    FALLBACK_ENABLED = True  # Fall back to general endpoint on failure

    @classmethod
    def get_endpoint_url(cls):
        """Get the URL for the active endpoint"""
        return ENDPOINTS.get(cls.ACTIVE_ENDPOINT, ENDPOINTS["scholarly"])

    @classmethod
    def get_journal_values_clause(cls):
        """Generate SPARQL VALUES clause for configured journals"""
        qids = " ".join([f"wd:{qid}" for qid in cls.JOURNALS.keys()])
        return f"VALUES ?journal {{ {qids} }}"

    @classmethod
    def get_journal_count(cls):
        return len(cls.JOURNALS)


def create_configuration_interface():
    """Create interactive interface for endpoint and journal configuration"""

    # Endpoint configuration section
    endpoint_html = """
    <div style='background-color: #E7ECEF; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #274C77;'>
    <h3 style='color: #274C77; margin-top: 0;'>🌐 SPARQL Endpoint Configuration</h3>
    <p>Select which Wikidata SPARQL endpoint to use for queries.</p>
    <div style='background-color: #A3CEF1; padding: 15px; border-radius: 8px; margin: 15px 0; border-left: 4px solid #6096BA;'>
        <p style='color: #274C77; margin: 0;'><strong>Scholarly endpoint</strong> (recommended): Optimized for bibliographic data queries</p>
        <p style='color: #274C77; margin: 5px 0 0 0;'><strong>General endpoint</strong>: Standard Wikidata query service</p>
    </div>
    </div>
    """
    display(HTML(endpoint_html))

    style = {'description_width': '120px'}
    layout = widgets.Layout(width='400px')

    endpoint_dropdown = widgets.Dropdown(
        options=[
            ('Scholarly (recommended)', 'scholarly'),
            ('General', 'general')
        ],
        value=Config.ACTIVE_ENDPOINT,
        description='Endpoint:',
        style=style,
        layout=layout
    )

    fallback_checkbox = widgets.Checkbox(
        value=Config.FALLBACK_ENABLED,
        description='Enable fallback to general endpoint on failure',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='400px')
    )

    endpoint_output = widgets.Output()

    def update_endpoint(change):
        Config.ACTIVE_ENDPOINT = change['new']
        with endpoint_output:
            clear_output()
            print(f"✓ Endpoint set to: {Config.ACTIVE_ENDPOINT}")
            print(f"   URL: {Config.get_endpoint_url()}")

    def update_fallback(change):
        Config.FALLBACK_ENABLED = change['new']
        with endpoint_output:
            clear_output()
            status = "enabled" if Config.FALLBACK_ENABLED else "disabled"
            print(f"✓ Fallback {status}")

    endpoint_dropdown.observe(update_endpoint, names='value')
    fallback_checkbox.observe(update_fallback, names='value')

    display(widgets.VBox([endpoint_dropdown, fallback_checkbox, endpoint_output]))

    # Journal configuration section
    journal_html = """
    <div style='background-color: #E7ECEF; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #274C77;'>
    <h3 style='color: #274C77; margin-top: 0;'>📚 Journal Configuration</h3>
    <p>Configure which journals to analyze for duplicate detection. The default set includes AAA journals and related anthropology publications.</p>
    <div style='background-color: #A3CEF1; padding: 15px; border-radius: 8px; margin: 15px 0; border-left: 4px solid #6096BA;'>
        <p style='color: #274C77; margin: 0; font-weight: bold;'>📋 Current journal set: {journal_count} journals configured</p>
    </div>
    </div>
    """.format(journal_count=Config.get_journal_count())
    display(HTML(journal_html))

    # Display current journals
    journal_list_html = "<div style='background-color: #E7ECEF; padding: 15px; border-radius: 10px; margin: 10px 0; max-height: 300px; overflow-y: auto;'>"
    journal_list_html += "<h4 style='color: #274C77; margin-top: 0;'>Configured Journals:</h4>"
    journal_list_html += "<ul style='color: #274C77; margin: 0; padding-left: 20px;'>"
    for qid, label in sorted(Config.JOURNALS.items(), key=lambda x: x[1]):
        journal_list_html += f"<li><code>{qid}</code> — {label}</li>"
    journal_list_html += "</ul></div>"
    display(HTML(journal_list_html))

    # Add custom journal widget
    add_journal_html = """
    <div style='background-color: #E7ECEF; padding: 15px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #6096BA;'>
    <h4 style='color: #274C77; margin-top: 0;'>➕ Add Custom Journal</h4>
    <p style='color: #274C77;'>Enter a Wikidata QID (e.g., Q12345) and label to add to the analysis.</p>
    </div>
    """
    display(HTML(add_journal_html))

    qid_input = widgets.Text(
        placeholder='Q12345',
        description='QID:',
        style={'description_width': '50px'},
        layout=widgets.Layout(width='200px')
    )

    label_input = widgets.Text(
        placeholder='Journal Name',
        description='Label:',
        style={'description_width': '50px'},
        layout=widgets.Layout(width='300px')
    )

    journal_output = widgets.Output()

    def add_journal(b):
        with journal_output:
            clear_output()
            qid = qid_input.value.strip().upper()
            label = label_input.value.strip()

            if not qid.startswith('Q'):
                qid = 'Q' + qid

            if qid and label:
                Config.JOURNALS[qid] = label
                print(f"✓ Added {qid}: {label}")
                print(f"Total journals: {Config.get_journal_count()}")
            else:
                print("⚠ Please enter both QID and label")

    def reset_journals(b):
        with journal_output:
            clear_output()
            Config.JOURNALS = DEFAULT_JOURNALS.copy()
            print(f"✓ Reset to default {Config.get_journal_count()} journals")

    add_button = widgets.Button(
        description='Add Journal',
        button_style='primary',
        style={'button_color': '#6096BA'}
    )
    add_button.on_click(add_journal)

    reset_button = widgets.Button(
        description='Reset to Defaults',
        button_style='warning',
        style={'button_color': '#8B8C89'}
    )
    reset_button.on_click(reset_journals)

    display(widgets.VBox([
        widgets.HBox([qid_input, label_input]),
        widgets.HBox([add_button, reset_button]),
        journal_output
    ]))

# Display configuration interface
create_configuration_interface()

## SPARQL Query Functions

*Core functions for executing SPARQL queries against Wikidata. These functions handle endpoint selection, fallback logic, query construction, and result parsing for both DOI-based and title-based duplicate detection.*

In [ ]:
def execute_sparql_query(query, timeout=60):
    """
    Execute a SPARQL query against the configured Wikidata endpoint.
    Falls back to general endpoint if scholarly fails and fallback is enabled.

    Args:
        query: SPARQL query string
        timeout: Query timeout in seconds

    Returns:
        pandas DataFrame with query results
    """
    endpoints_to_try = [Config.ACTIVE_ENDPOINT]

    # Add fallback endpoint if enabled
    if Config.FALLBACK_ENABLED and Config.ACTIVE_ENDPOINT == "scholarly":
        endpoints_to_try.append("general")
    elif Config.FALLBACK_ENABLED and Config.ACTIVE_ENDPOINT == "general":
        endpoints_to_try.append("scholarly")

    last_error = None

    for endpoint_name in endpoints_to_try:
        endpoint_url = ENDPOINTS[endpoint_name]

        sparql = SPARQLWrapper(endpoint_url)
        sparql.setQuery(query)
        sparql.setReturnFormat(JSON)
        sparql.setTimeout(timeout)
        sparql.agent = "WikidataDuplicateDetector/1.0 (https://github.com/MattArtzAnthro; anthropology research)"

        try:
            if endpoint_name != Config.ACTIVE_ENDPOINT:
                print(f"   ↳ Falling back to {endpoint_name} endpoint...")

            results = sparql.query().convert()
            bindings = results["results"]["bindings"]

            if not bindings:
                return pd.DataFrame()

            # Extract variable names
            columns = list(bindings[0].keys())

            # Parse results
            data = []
            for row in bindings:
                row_data = {}
                for col in columns:
                    if col in row:
                        row_data[col] = row[col].get("value", "")
                    else:
                        row_data[col] = ""
                data.append(row_data)

            return pd.DataFrame(data)

        except Exception as e:
            last_error = e
            if endpoint_name == endpoints_to_try[-1]:
                print(f"❌ Query error: {str(e)}")
            continue

    return pd.DataFrame()


def build_doi_duplicates_query():
    """
    Build SPARQL query to find DOI-based duplicates within configured journals.

    Returns:
        SPARQL query string
    """
    values_clause = Config.get_journal_values_clause()

    query = f"""
    SELECT ?journal ?journalLabel ?doi (COUNT(?item) AS ?count) WHERE {{
      {values_clause}
      ?item wdt:P1433 ?journal ;
            wdt:P356 ?doi .
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    GROUP BY ?journal ?journalLabel ?doi
    HAVING (COUNT(?item) > 1)
    ORDER BY DESC(?count) ?journalLabel
    """
    return query


def build_title_duplicates_query(journal_qid):
    """
    Build SPARQL query to find title-based duplicates for a specific journal.

    Args:
        journal_qid: Wikidata QID for the journal (e.g., "Q15755065")

    Returns:
        SPARQL query string
    """
    query = f"""
    SELECT ?titleNorm (COUNT(?item) AS ?count) WHERE {{
      ?item wdt:P1433 wd:{journal_qid} ;
            wdt:P31 wd:Q13442814 ;
            wdt:P1476 ?title .
      BIND(LCASE(STR(?title)) AS ?titleNorm)
    }}
    GROUP BY ?titleNorm
    HAVING (COUNT(?item) > 1)
    ORDER BY DESC(?count)
    LIMIT 200
    """
    return query


def build_doi_inspection_query(doi):
    """
    Build SPARQL query to inspect all items sharing a specific DOI.

    Args:
        doi: DOI string to inspect

    Returns:
        SPARQL query string
    """
    query = f"""
    SELECT ?item ?itemLabel ?title ?date ?instanceOf ?instanceOfLabel ?journal ?journalLabel WHERE {{
      ?item wdt:P356 "{doi}" .
      OPTIONAL {{ ?item wdt:P1476 ?title }}
      OPTIONAL {{ ?item wdt:P577 ?date }}
      OPTIONAL {{ ?item wdt:P31 ?instanceOf }}
      OPTIONAL {{ ?item wdt:P1433 ?journal }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """
    return query


def build_title_inspection_query(journal_qid, normalized_title):
    """
    Build SPARQL query to inspect all items sharing a normalized title.

    Args:
        journal_qid: Wikidata QID for the journal
        normalized_title: Lowercase normalized title string

    Returns:
        SPARQL query string
    """
    # Escape quotes in title
    escaped_title = normalized_title.replace('"', '\\"')

    query = f"""
    SELECT ?item ?itemLabel ?title ?date ?doi WHERE {{
      ?item wdt:P1433 wd:{journal_qid} ;
            wdt:P31 wd:Q13442814 ;
            wdt:P1476 ?title .
      BIND(LCASE(STR(?title)) AS ?titleNorm)
      FILTER(?titleNorm = "{escaped_title}")
      OPTIONAL {{ ?item wdt:P577 ?date }}
      OPTIONAL {{ ?item wdt:P356 ?doi }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    ORDER BY ?date
    """
    return query


print("✓ SPARQL query functions loaded")
print(f"✓ Active endpoint: {Config.ACTIVE_ENDPOINT} ({Config.get_endpoint_url()})")
print(f"✓ Fallback enabled: {Config.FALLBACK_ENABLED}")

## DOI-Based Duplicate Detection

*Identify definitive duplicates by finding multiple Wikidata items sharing the same DOI. DOIs are unique identifiers, so any DOI appearing on multiple items represents a true duplicate requiring merger or cleanup.*

In [ ]:
def run_doi_duplicate_detection():
    """
    Run DOI-based duplicate detection across configured journals.

    Returns:
        DataFrame with duplicate DOIs and counts
    """
    print("🔍 Running DOI-based duplicate detection...")
    print(f"   Endpoint: {Config.ACTIVE_ENDPOINT} ({Config.get_endpoint_url()})")
    print(f"   Analyzing {Config.get_journal_count()} journals")
    print()

    query = build_doi_duplicates_query()
    df = execute_sparql_query(query)

    if df.empty:
        print("✓ No DOI duplicates found!")
        return df

    # Clean up journal URIs to QIDs
    if 'journal' in df.columns:
        df['journal_qid'] = df['journal'].apply(lambda x: x.split('/')[-1] if pd.notna(x) else '')
        df['journal_name'] = df['journal_qid'].apply(lambda x: Config.JOURNALS.get(x, x))

    # Convert count to int
    if 'count' in df.columns:
        df['count'] = pd.to_numeric(df['count'], errors='coerce').fillna(0).astype(int)

    print(f"⚠ Found {len(df)} DOI duplicates")
    print()

    # Summary by journal
    if 'journal_name' in df.columns:
        summary = df.groupby('journal_name').size().sort_values(ascending=False)
        print("Duplicates by journal:")
        for journal, count in summary.items():
            print(f"   {journal}: {count}")

    return df

# Store results globally for later use
doi_duplicates_df = None

def display_doi_detection_interface():
    """Create interactive interface for DOI duplicate detection"""
    global doi_duplicates_df

    instructions_html = """
    <div style='background-color: #E7ECEF; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #274C77;'>
    <h3 style='color: #274C77; margin-top: 0;'>🔍 DOI-Based Duplicate Detection</h3>
    <p>Find articles that share the same DOI across multiple Wikidata items. Since DOIs are unique identifiers, these represent definitive duplicates.</p>
    <div style='background-color: #A3CEF1; padding: 15px; border-radius: 8px; margin: 15px 0; border-left: 4px solid #6096BA;'>
        <p style='color: #274C77; margin: 0;'>⚡ <strong>Note:</strong> This query searches all configured journals simultaneously. Large journal sets may take 30-60 seconds.</p>
    </div>
    </div>
    """
    display(HTML(instructions_html))

    output = widgets.Output()

    def run_detection(b):
        global doi_duplicates_df
        with output:
            clear_output()
            doi_duplicates_df = run_doi_duplicate_detection()

            if not doi_duplicates_df.empty:
                print("\n" + "="*60)
                print("Results preview (first 20 rows):")
                print("="*60)
                display_cols = ['journal_name', 'doi', 'count'] if 'journal_name' in doi_duplicates_df.columns else doi_duplicates_df.columns.tolist()
                display(doi_duplicates_df[display_cols].head(20))

    run_button = widgets.Button(
        description='Run DOI Detection',
        button_style='primary',
        style={'button_color': '#6096BA'},
        icon='search'
    )
    run_button.on_click(run_detection)

    display(run_button)
    display(output)

# Display interface
display_doi_detection_interface()

## Inspect DOI Duplicates

*Examine specific duplicate pairs to understand how they differ. This reveals whether duplicates resulted from different typing (P31), missing data, or other modeling inconsistencies.*

In [ ]:
def inspect_doi_duplicate(doi):
    """
    Inspect all Wikidata items sharing a specific DOI.

    Args:
        doi: DOI string to inspect

    Returns:
        DataFrame with item details
    """
    print(f"🔎 Inspecting DOI: {doi}")
    print(f"   Endpoint: {Config.ACTIVE_ENDPOINT}")
    print()

    query = build_doi_inspection_query(doi)
    df = execute_sparql_query(query)

    if df.empty:
        print("No items found with this DOI")
        return df

    # Clean up URIs to QIDs
    if 'item' in df.columns:
        df['item_qid'] = df['item'].apply(lambda x: x.split('/')[-1] if pd.notna(x) else '')
        df['item_url'] = df['item_qid'].apply(lambda x: f"https://www.wikidata.org/wiki/{x}")

    if 'instanceOf' in df.columns:
        df['type_qid'] = df['instanceOf'].apply(lambda x: x.split('/')[-1] if pd.notna(x) else '')

    print(f"Found {len(df)} items sharing this DOI:")
    print()

    for idx, row in df.iterrows():
        print(f"Item {idx + 1}: {row.get('item_qid', 'N/A')}")
        print(f"   Title: {row.get('title', 'N/A')}")
        print(f"   Date: {row.get('date', 'N/A')}")
        print(f"   Type: {row.get('instanceOfLabel', 'N/A')} ({row.get('type_qid', 'N/A')})")
        print(f"   URL: {row.get('item_url', 'N/A')}")
        print()

    return df


def display_doi_inspection_interface():
    """Create interactive interface for DOI inspection"""

    instructions_html = """
    <div style='background-color: #E7ECEF; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #274C77;'>
    <h3 style='color: #274C77; margin-top: 0;'>🔎 Inspect DOI Duplicate</h3>
    <p>Enter a DOI from the detection results to see all Wikidata items sharing that identifier. This reveals differences in typing, dates, and other properties.</p>
    </div>
    """
    display(HTML(instructions_html))

    doi_input = widgets.Text(
        placeholder='10.1111/ANHU.12539',
        description='DOI:',
        style={'description_width': '50px'},
        layout=widgets.Layout(width='400px')
    )

    output = widgets.Output()

    def run_inspection(b):
        with output:
            clear_output()
            doi = doi_input.value.strip()
            if doi:
                inspect_doi_duplicate(doi)
            else:
                print("⚠ Please enter a DOI")

    inspect_button = widgets.Button(
        description='Inspect DOI',
        button_style='primary',
        style={'button_color': '#6096BA'},
        icon='search'
    )
    inspect_button.on_click(run_inspection)

    display(widgets.HBox([doi_input, inspect_button]))
    display(output)

# Display interface
display_doi_inspection_interface()

## Title-Based Duplicate Detection

*Identify potential duplicates by finding articles with matching normalized (lowercase) titles within a specific journal. Title matches require manual review since different articles may legitimately share titles.*

In [ ]:
def run_title_duplicate_detection(journal_qid, journal_name=None):
    """
    Run title-based duplicate detection for a specific journal.

    Args:
        journal_qid: Wikidata QID for the journal
        journal_name: Human-readable journal name (optional)

    Returns:
        DataFrame with duplicate titles and counts
    """
    if journal_name is None:
        journal_name = Config.JOURNALS.get(journal_qid, journal_qid)

    print(f"🔍 Running title-based duplicate detection...")
    print(f"   Endpoint: {Config.ACTIVE_ENDPOINT} ({Config.get_endpoint_url()})")
    print(f"   Journal: {journal_name} ({journal_qid})")
    print()

    query = build_title_duplicates_query(journal_qid)
    df = execute_sparql_query(query)

    if df.empty:
        print("✓ No title duplicates found!")
        return df

    # Convert count to int
    if 'count' in df.columns:
        df['count'] = pd.to_numeric(df['count'], errors='coerce').fillna(0).astype(int)

    # Add journal info
    df['journal_qid'] = journal_qid
    df['journal_name'] = journal_name

    print(f"⚠ Found {len(df)} potential title duplicates")
    print()
    print("Note: Title matches require manual review—different articles may share titles.")

    return df

# Store results globally
title_duplicates_df = None

def display_title_detection_interface():
    """Create interactive interface for title duplicate detection"""
    global title_duplicates_df

    instructions_html = """
    <div style='background-color: #E7ECEF; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #274C77;'>
    <h3 style='color: #274C77; margin-top: 0;'>📝 Title-Based Duplicate Detection</h3>
    <p>Find articles with matching normalized titles within a single journal. Select a journal from your configured set to analyze.</p>
    <div style='background-color: #A3CEF1; padding: 15px; border-radius: 8px; margin: 15px 0; border-left: 4px solid #6096BA;'>
        <p style='color: #274C77; margin: 0;'>⚠️ <strong>Note:</strong> Title matches are candidates for review, not definitive duplicates. Different articles may legitimately share titles (e.g., "Introduction", "Book Reviews").</p>
    </div>
    </div>
    """
    display(HTML(instructions_html))

    # Create dropdown with journal options
    journal_options = [(f"{name} ({qid})", qid) for qid, name in sorted(Config.JOURNALS.items(), key=lambda x: x[1])]

    journal_dropdown = widgets.Dropdown(
        options=journal_options,
        description='Journal:',
        style={'description_width': '80px'},
        layout=widgets.Layout(width='450px')
    )

    output = widgets.Output()

    def run_detection(b):
        global title_duplicates_df
        with output:
            clear_output()
            journal_qid = journal_dropdown.value
            title_duplicates_df = run_title_duplicate_detection(journal_qid)

            if not title_duplicates_df.empty:
                print("\n" + "="*60)
                print("Results preview (first 20 rows):")
                print("="*60)
                display(title_duplicates_df[['titleNorm', 'count']].head(20))

    run_button = widgets.Button(
        description='Run Title Detection',
        button_style='primary',
        style={'button_color': '#6096BA'},
        icon='search'
    )
    run_button.on_click(run_detection)

    display(widgets.HBox([journal_dropdown, run_button]))
    display(output)

# Display interface
display_title_detection_interface()

## Inspect Title Duplicates

*Examine specific title matches to determine whether they represent true duplicates or distinct articles. Compare DOIs, publication dates, and other properties to make an informed judgment.*

In [ ]:
def inspect_title_duplicate(journal_qid, normalized_title):
    """
    Inspect all Wikidata items sharing a normalized title in a journal.

    Args:
        journal_qid: Wikidata QID for the journal
        normalized_title: Lowercase normalized title string

    Returns:
        DataFrame with item details
    """
    journal_name = Config.JOURNALS.get(journal_qid, journal_qid)

    print(f"🔎 Inspecting title match in {journal_name}")
    print(f"   Endpoint: {Config.ACTIVE_ENDPOINT}")
    print(f"   Title: \"{normalized_title}\"")
    print()

    query = build_title_inspection_query(journal_qid, normalized_title)
    df = execute_sparql_query(query)

    if df.empty:
        print("No items found with this title")
        return df

    # Clean up URIs to QIDs
    if 'item' in df.columns:
        df['item_qid'] = df['item'].apply(lambda x: x.split('/')[-1] if pd.notna(x) else '')
        df['item_url'] = df['item_qid'].apply(lambda x: f"https://www.wikidata.org/wiki/{x}")

    print(f"Found {len(df)} items sharing this title:")
    print()

    # Check if DOIs match (indicates true duplicate)
    dois = df['doi'].dropna().unique() if 'doi' in df.columns else []
    if len(dois) == 1:
        print("⚠️ All items share the same DOI — likely true duplicate")
        print()
    elif len(dois) > 1:
        print("ℹ️ Items have different DOIs — may be distinct articles")
        print()
    elif len(dois) == 0:
        print("ℹ️ No DOIs present — requires manual verification")
        print()

    for idx, row in df.iterrows():
        print(f"Item {idx + 1}: {row.get('item_qid', 'N/A')}")
        print(f"   Title: {row.get('title', 'N/A')}")
        print(f"   Date: {row.get('date', 'N/A')}")
        print(f"   DOI: {row.get('doi', 'N/A')}")
        print(f"   URL: {row.get('item_url', 'N/A')}")
        print()

    return df


def display_title_inspection_interface():
    """Create interactive interface for title inspection"""

    instructions_html = """
    <div style='background-color: #E7ECEF; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #274C77;'>
    <h3 style='color: #274C77; margin-top: 0;'>🔎 Inspect Title Duplicate</h3>
    <p>Enter a normalized title from the detection results and select the journal to see all matching items. Compare DOIs and dates to determine if these are true duplicates.</p>
    </div>
    """
    display(HTML(instructions_html))

    # Journal dropdown
    journal_options = [(f"{name} ({qid})", qid) for qid, name in sorted(Config.JOURNALS.items(), key=lambda x: x[1])]

    journal_dropdown = widgets.Dropdown(
        options=journal_options,
        description='Journal:',
        style={'description_width': '80px'},
        layout=widgets.Layout(width='450px')
    )

    title_input = widgets.Text(
        placeholder='enter normalized title (lowercase)',
        description='Title:',
        style={'description_width': '80px'},
        layout=widgets.Layout(width='450px')
    )

    output = widgets.Output()

    def run_inspection(b):
        with output:
            clear_output()
            journal_qid = journal_dropdown.value
            title = title_input.value.strip().lower()
            if title:
                inspect_title_duplicate(journal_qid, title)
            else:
                print("⚠ Please enter a title")

    inspect_button = widgets.Button(
        description='Inspect Title',
        button_style='primary',
        style={'button_color': '#6096BA'},
        icon='search'
    )
    inspect_button.on_click(run_inspection)

    display(widgets.VBox([
        journal_dropdown,
        title_input,
        inspect_button
    ]))
    display(output)

# Display interface
display_title_inspection_interface()

## Export Results

*Export detection results to CSV files for documentation, manual review, or batch processing in OpenRefine. Exports include timestamps and journal metadata for traceability.*

In [ ]:
def export_results():
    """
    Export detection results to CSV files.
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    exports = []

    # Export DOI duplicates
    if doi_duplicates_df is not None and not doi_duplicates_df.empty:
        doi_path = f"{Config.OUTPUT_PATH}doi_duplicates_{timestamp}.csv"
        export_cols = ['journal_name', 'journal_qid', 'doi', 'count']
        available_cols = [c for c in export_cols if c in doi_duplicates_df.columns]
        doi_duplicates_df[available_cols].to_csv(doi_path, index=False)
        exports.append(('DOI duplicates', doi_path, len(doi_duplicates_df)))

    # Export title duplicates
    if title_duplicates_df is not None and not title_duplicates_df.empty:
        title_path = f"{Config.OUTPUT_PATH}title_duplicates_{timestamp}.csv"
        export_cols = ['journal_name', 'journal_qid', 'titleNorm', 'count']
        available_cols = [c for c in export_cols if c in title_duplicates_df.columns]
        title_duplicates_df[available_cols].to_csv(title_path, index=False)
        exports.append(('Title duplicates', title_path, len(title_duplicates_df)))

    if exports:
        print("✓ Export complete!")
        print()
        for name, path, count in exports:
            print(f"   {name}: {path} ({count} records)")
    else:
        print("⚠ No results to export. Run detection first.")

    return exports


def display_export_interface():
    """Create interactive interface for exporting results"""

    instructions_html = """
    <div style='background-color: #E7ECEF; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #274C77;'>
    <h3 style='color: #274C77; margin-top: 0;'>💾 Export Results</h3>
    <p>Export detection results to CSV files for documentation or further processing. Files are saved to the output directory with timestamps.</p>
    <div style='background-color: #A3CEF1; padding: 15px; border-radius: 8px; margin: 15px 0; border-left: 4px solid #6096BA;'>
        <p style='color: #274C77; margin: 0;'>📁 <strong>Output directory:</strong> {output_path}</p>
    </div>
    </div>
    """.format(output_path=Config.OUTPUT_PATH)
    display(HTML(instructions_html))

    output = widgets.Output()

    def run_export(b):
        with output:
            clear_output()
            export_results()

    export_button = widgets.Button(
        description='Export to CSV',
        button_style='primary',
        style={'button_color': '#6096BA'},
        icon='download'
    )
    export_button.on_click(run_export)

    display(export_button)
    display(output)

# Display interface
display_export_interface()

## Download Results

*Download exported CSV files to your local machine. In Google Colab, this triggers a browser download dialog.*

In [ ]:
import glob

def download_results():
    """Download all CSV files from output directory."""
    csv_files = glob.glob(f"{Config.OUTPUT_PATH}*.csv")

    if not csv_files:
        print("No CSV files found. Run export first.")
        return

    print(f"Downloading {len(csv_files)} file(s)...")
    print()

    for filepath in csv_files:
        filename = os.path.basename(filepath)
        print(f"   {filename}")

    try:
        from google.colab import files
        for filepath in csv_files:
            files.download(filepath)
        print("\nDownload complete!")
    except ImportError:
        print("\n(Files saved to output directory)")

def display_download_interface():
    """Create interactive interface for downloading results"""
    display(HTML(f"""
    <div style='background-color: #E7ECEF; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #274C77;'>
    <h3 style='color: #274C77; margin-top: 0;'>Download Results</h3>
    <p>Download exported CSV files for review or import into OpenRefine.</p>
    </div>
    """))

    output = widgets.Output()

    def run_download(b):
        with output:
            clear_output()
            download_results()

    download_button = widgets.Button(
        description='Download CSV Files',
        button_style='success',
        style={'button_color': '#6096BA'},
        icon='download'
    )
    download_button.on_click(run_download)

    display(download_button)
    display(output)

display_download_interface()

## Custom SPARQL Queries

*Run custom SPARQL queries against the configured Wikidata endpoint for additional analysis beyond the standard detection patterns. Useful for investigating specific edge cases or exploring related data quality issues.*

In [ ]:
def display_custom_query_interface():
    """Create interface for running custom SPARQL queries"""

    instructions_html = f"""
    <div style='background-color: #E7ECEF; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #274C77;'>
    <h3 style='color: #274C77; margin-top: 0;'>⚙️ Custom SPARQL Query</h3>
    <p>Run custom SPARQL queries against Wikidata for additional analysis. Results are displayed as a table and can be exported.</p>
    <div style='background-color: #A3CEF1; padding: 15px; border-radius: 8px; margin: 15px 0; border-left: 4px solid #6096BA;'>
        <p style='color: #274C77; margin: 0;'>🌐 <strong>Active endpoint:</strong> {Config.ACTIVE_ENDPOINT} ({Config.get_endpoint_url()})</p>
    </div>
    </div>
    """
    display(HTML(instructions_html))

    # Example query
    example_query = """SELECT ?item ?itemLabel ?doi WHERE {
  ?item wdt:P1433 wd:Q15755065 ;
        wdt:P356 ?doi .
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
LIMIT 10"""

    query_textarea = widgets.Textarea(
        value=example_query,
        placeholder='Enter SPARQL query',
        layout=widgets.Layout(width='100%', height='200px')
    )

    output = widgets.Output()

    def run_query(b):
        with output:
            clear_output()
            query = query_textarea.value.strip()
            if query:
                print(f"🔍 Executing query on {Config.ACTIVE_ENDPOINT} endpoint...")
                print()
                df = execute_sparql_query(query)
                if not df.empty:
                    print(f"✓ Found {len(df)} results")
                    print()
                    display(df)
                else:
                    print("No results returned")
            else:
                print("⚠ Please enter a query")

    run_button = widgets.Button(
        description='Run Query',
        button_style='primary',
        style={'button_color': '#6096BA'},
        icon='play'
    )
    run_button.on_click(run_query)

    display(query_textarea)
    display(run_button)
    display(output)

# Display interface
display_custom_query_interface()